In [ ]:
# Install required packages (auto-skipped if already installed)
import importlib
if importlib.util.find_spec('qiskit') is None:
    !pip install -q qiskit qiskit-aer qiskit-ibm-runtime pylatexenc numpy
else:
    print("\u2713 Packages already installed")

# To run on real quantum hardware, uncomment and fill in your credentials:
# from qiskit_ibm_runtime import QiskitRuntimeService
# QiskitRuntimeService.save_account(
#     token="<your-api-key>",
#     instance="<your-crn>",
#     overwrite=True
# )

# مراقبة مهمة أو إلغاؤها

اطّلع على قائمة أحمال عملك في [صفحة Workloads](https://quantum.cloud.ibm.com/workloads).

## عرض حالة المهمة
انتقل إلى [جدول Workloads](https://quantum.cloud.ibm.com/workloads) وتحقق من عمود Status لمعرفة ما إذا كانت المهمة قد اكتملت أو فشلت.

## عرض الاستخدام المتبقي
انتقل إلى [جدول Instances](https://quantum.cloud.ibm.com/instances) واختر التبويب المرتبط بالخطة التي تريد معرفة استخدامها المتبقي. سيظهر لك إجمالي الوقت المُستخدَم والوقت المتبقي في خطتك.

## عرض مقاييس عدد المهام وأحمال العمل المُرسَلة
انتقل إلى [صفحة Analytics](https://quantum.cloud.ibm.com/analytics) لترى الإجمالي الكلي للمهام المُرسَلة، إضافةً إلى عدد أحمال عمل الدُّفعات (batch) وأحمال عمل الجلسات (session). لاحظ أنك لا تستطيع الاطلاع على صفحة Analytics إلا للحسابات التي تمتلكها أو تديرها.

## مراقبة مهمة
استخدم نسخة المهمة (job instance) للتحقق من حالتها أو استرداد نتائجها عبر استدعاء الأمر المناسب:

|                               |                                                                                                                                                                                                              |
| ----------------------------- | ------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------ |
| job.result()                  | اعرض نتائج المهمة فور اكتمالها. تتوفر النتائج بعد انتهاء المهمة، لذا فإن `job.result()` استدعاءٌ محجوب (blocking call) حتى تكتمل المهمة.                               |
| job.job\_id()                 | أعِد المعرّف الفريد للمهمة. استرداد النتائج لاحقاً يستلزم معرفة هذا المعرّف، لذا يُنصح بحفظ معرّفات المهام التي قد تحتاج إلى استردادها لاحقاً. |
| job.status()                  | تحقق من حالة المهمة.                                                                                                                                                                                        |
| job = service.job(\<job\_id>) | استرجع مهمة سبق أن أرسلتها. يستلزم هذا الاستدعاء معرّف المهمة.                                                                                                                                      |

<span id="retrieve-later"></span>
## استرداد نتائج المهمة في وقت لاحق
استدعِ `service.job(\<job\_id>)` لاسترداد مهمة أرسلتها سابقاً. إن لم يكن لديك معرّف المهمة، أو أردت استرداد عدة مهام دفعةً واحدة — بما في ذلك المهام من وحدات المعالجة الكمية (QPUs) المُتقاعَدة — فاستخدم `service.jobs()` مع مرشّحات اختيارية. راجع [QiskitRuntimeService.jobs](../api/qiskit-ibm-runtime/qiskit-runtime-service#jobs).

> **Note:** تُعيد `service.jobs()` أيضاً المهام التي نُفِّذت من حزمة `qiskit-ibm-provider` المُهمَلة. أما المهام التي أُرسلت عبر الحزمة الأقدم (المُهمَلة أيضاً) `qiskit-ibmq-provider` فلم تعد متاحة.

### مثال
يُعيد هذا المثال أحدث 10 مهام runtime نُفِّذت على `my_backend`:

In [1]:
# This cell is hidden from users
from qiskit import QuantumCircuit
from qiskit.circuit import Parameter
from qiskit.transpiler import generate_preset_pass_manager

from qiskit_ibm_runtime import QiskitRuntimeService, SamplerV2
import numpy as np


my_backend = "ibm_torino"
service = QiskitRuntimeService()
# backend = service.backend(my_backend)
backend = service.least_busy()

# Define two circuits, each with one parameter with two parameters.
circuit = QuantumCircuit(2)
circuit.h(0)
circuit.cx(0, 1)
circuit.ry(Parameter("a"), 0)
circuit.cx(0, 1)
circuit.h(0)
circuit.measure_all()


pm = generate_preset_pass_manager(optimization_level=1, backend=backend)
transpiled_circuit = pm.run(circuit)

params = np.random.uniform(size=(2, 3)).T

sampler_pub = (transpiled_circuit, params)

# Instantiate the new estimator object, then run the transpiled circuit
# using the set of parameters and observables.
sampler = SamplerV2(mode=backend)
job = sampler.run([sampler_pub], shots=4)
print(job.job_id())

d305ck0ocacs73ajagvg


In [2]:
result = job.result()


spans = job.result().metadata["execution"]["execution_spans"]
print(spans)

ExecutionSpans([DoubleSliceSpan(<start='2025-09-09 16:31:16', stop='2025-09-09 16:31:16', size=24>)])


In [3]:
params = np.random.uniform(size=(2, 3))
params

array([[0.2260416 , 0.8747859 , 0.44361995],
       [0.94700856, 0.96826017, 0.98426562]])

In [4]:
mask = spans[0].mask(0)
mask

array([[[ True,  True,  True,  True],
        [ True,  True,  True,  True]],

       [[ True,  True,  True,  True],
        [ True,  True,  True,  True]],

       [[ True,  True,  True,  True],
        [ True,  True,  True,  True]]])

In [ ]:
from qiskit_ibm_runtime import QiskitRuntimeService

# Initialize the account first.
service = QiskitRuntimeService()
# Use `limit` to retrieve a specific number of jobs. The default `limit` is 10.
service.jobs(backend_name=my_backend)

## Cancel a job

You can cancel a job from the IBM Quantum Platform dashboard either on the Workloads page or the details page for a specific workload. On the Workloads page, click the overflow menu at the end of the row for that workload, and select Cancel. If you are on the details page for a specific workload, use the Actions dropdown at the top of the page, and select Cancel.

In Qiskit, use `job.cancel()` to cancel a job.

<span id="execution-spans"></span>
## View Sampler execution spans

The results of [`SamplerV2`](/docs/api/qiskit-ibm-runtime/sampler-v2) jobs executed in Qiskit Runtime contain execution timing information in their metadata.
This timing information can be used to place upper and lower timestamp bounds on when particular shots were executed on the QPU.
Shots are grouped into [`ExecutionSpan`](/docs/api/qiskit-ibm-runtime/execution-span-execution-span) objects, each of which indicates a start time, a stop time, and a specification of which shots were collected in the span.

An execution span specifies which data was executed during its window by providing an [`ExecutionSpan.mask`](/docs/api/qiskit-ibm-runtime/execution-span-execution-span#mask) method. This method, given any [Primitive Unified Block (PUB)](/docs/guides/primitive-input-output) index, returns a boolean mask that is `True` for all shots executed during its window. PUBs are indexed by the order in which they were given to the Sampler run call. If, for example, a PUB has shape `(2, 3)` and was run with four shots, then the mask's shape is `(2, 3, 4)`. See the [execution_span](/docs/api/qiskit-ibm-runtime/execution-span) API page for full details.

Example:

To view execution span information, review the metadata of the result returned by `SamplerV2`, which comes in the form of an `ExecutionSpans` object. This object is a list-like container containing instances of subclasses of `ExecutionSpan`, such as `SliceSpan`:

In [6]:
from qiskit.primitives import BitArray

# Get the mask of the 1st PUB for the 0th span.
mask = spans[0].mask(0)

# Decide whether the 0th shot of parameter set (1, 2) occurred in this span.
in_this_span = mask[2, 1, 0]

# Create a new bit array containing only the PUB-1 data collected during this span.
bits = result[0].data.meas
filtered_data = BitArray(bits.array[mask], bits.num_bits)

Execution spans can be filtered to include information pertaining to specific PUBs, selected by their indices:

In [7]:
# take the subset of spans that reference data in PUBs 0 or 2
spans.filter_by_pub([0, 2])

ExecutionSpans([DoubleSliceSpan(<start='2025-09-09 16:31:16', stop='2025-09-09 16:31:16', size=24>)])

View global information about the collection of execution spans:

In [8]:
print("Number of execution spans:", len(spans))
print("  Start of the first span:", spans.start)
print("     End of the last span:", spans.stop)
print("       Total duration (s):", spans.duration)

Number of execution spans: 1
  Start of the first span: 2025-09-09 16:31:16.320568
     End of the last span: 2025-09-09 16:31:16.865858
       Total duration (s): 0.54529


Extract and inspect a particular span:

In [9]:
spans.sort()
print(" Start of first span:", spans[0].start)
print("   End of first span:", spans[0].stop)
print("#shots in first span:", spans[0].size)

 Start of first span: 2025-09-09 16:31:16.320568
   End of first span: 2025-09-09 16:31:16.865858
#shots in first span: 24


## إلغاء مهمة
يمكنك إلغاء مهمة من لوحة تحكم IBM Quantum Platform إما من صفحة Workloads أو من صفحة التفاصيل لحمل عمل محدد. في صفحة Workloads، انقر على قائمة الفائض (overflow menu) في نهاية صف حمل العمل واختر Cancel. أما إذا كنت في صفحة التفاصيل لحمل عمل محدد، فاستخدم القائمة المنسدلة Actions في أعلى الصفحة واختر Cancel.

في Qiskit، استخدم `job.cancel()` لإلغاء مهمة.

<span id="execution-spans"></span>
## عرض نطاقات تنفيذ Sampler
تحتوي نتائج مهام [`SamplerV2`](https://docs.quantum.ibm.com/api/qiskit-ibm-runtime/sampler-v2) المُنفَّذة في Qiskit Runtime على معلومات توقيت التنفيذ ضمن بياناتها الوصفية (metadata).
يمكن استخدام هذه المعلومات لتحديد حدود زمنية عليا ودنيا لوقت تنفيذ لقطات (shots) معينة على وحدة المعالجة الكمية (QPU).
تُجمَّع اللقطات في كائنات [`ExecutionSpan`](https://docs.quantum.ibm.com/api/qiskit-ibm-runtime/execution-span-execution-span)، يُحدد كل منها وقت بدء وانتهاء، إضافةً إلى تفاصيل اللقطات التي جُمعت خلال ذلك النطاق.

يُحدد نطاق التنفيذ البيانات التي نُفِّذت خلال نافذته عبر توفير طريقة [`ExecutionSpan.mask`](https://docs.quantum.ibm.com/api/qiskit-ibm-runtime/execution-span-execution-span#mask). تُعيد هذه الطريقة، بالنسبة لأي فهرس [Primitive Unified Block (PUB)](/guides/primitive-input-output)، قناعاً (mask) منطقياً يكون `True` لجميع اللقطات المُنفَّذة خلال تلك النافذة. تُفهرَس وحدات PUB وفق الترتيب الذي أُعطيت به لاستدعاء Sampler run. فمثلاً، إذا كانت لوحدة PUB شكل `(2, 3)` ونُفِّذت بأربع لقطات، فإن شكل القناع يكون `(2, 3, 4)`. راجع صفحة API الخاصة بـ [execution_span](https://docs.quantum.ibm.com/api/qiskit-ibm-runtime/execution-span) للاطلاع على التفاصيل الكاملة.

مثال:
لعرض معلومات نطاق التنفيذ، راجع البيانات الوصفية للنتيجة التي يُعيدها `SamplerV2`، والتي تأتي في شكل كائن `ExecutionSpans`. هذا الكائن عبارة عن حاوية شبيهة بالقائمة تحتوي على نسخ من الفئات الفرعية لـ `ExecutionSpan`، مثل `SliceSpan`: